In [ ]:
import os
import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
from mmap_ninja import RaggedMmap

from training.networks_stylegan3 import Generator
from training.networks_stylegan2 import Discriminator
from torch_utils.ops import conv2d_gradfix

DEVICE = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
H = W = 64
CHANNELS = 1
BATCH_SIZE = 32
EPOCHS = 10
LR = 2e-4
BETAS = (0.0, 0.99)
GAMMA_R1 = 10.0
D_REG_INTERVAL = 16
SPARSITY = 0.2
Z_DIM = 512
LABEL_NOISE = 0.0

BASE_DIR = os.environ.get('NAVIERSTOKES_BASE_DIR', '/pscratch/sd/d/dpark1/NSData')
NAME = os.environ.get('NAVIERSTOKES_RUN_NAME', '100k')
MMAP_PATH = os.path.join(BASE_DIR, NAME)
MEAN, STD = 0.0, 2.4036
TARGET_TRAIN_PAIRS = 4900
VAL_PAIRS = 4900
TRAIN_SEED = 0
VAL_SEED = 12345

assert os.path.isdir(MMAP_PATH), f"Navier-Stokes memmap directory not found: {MMAP_PATH}"


def build_fixed_item_splits(mem, target_train_pairs, val_pairs=4900, train_seed=0, val_seed=12345):
    num_items_total = len(mem)
    assert num_items_total > 0
    T = int(mem[0].shape[0])
    assert T >= 2
    pairs_per_item = T - 1
    items_for_val = int(np.ceil(val_pairs / pairs_per_item))
    items_for_train = int(np.ceil(target_train_pairs / pairs_per_item))
    rng_val = np.random.default_rng(val_seed)
    val_items = np.sort(rng_val.choice(num_items_total, size=items_for_val, replace=False))
    remaining = np.setdiff1d(np.arange(num_items_total), val_items, assume_unique=True)
    assert len(remaining) >= items_for_train
    rng_tr = np.random.default_rng(train_seed)
    train_items = np.sort(rng_tr.choice(remaining, size=items_for_train, replace=False))

    def pairs_from_items(items):
        pairs = [(int(it), int(t)) for it in items for t in range(T - 1)]
        return np.asarray(pairs, dtype=np.int64)

    train_pairs = pairs_from_items(train_items)[:target_train_pairs]
    val_pairs = pairs_from_items(val_items)[:val_pairs]
    return train_pairs, val_pairs, T


class NavierStokesSparseNextStep(Dataset):
    def __init__(self, memmap, index_pairs, sparsity=0.2, normalize=True, mean=0.0, std=1.0, fixed_mask_per_sample=True, mask_seed=0):
        self.mem = memmap
        self.index_pairs = np.asarray(index_pairs, dtype=np.int64)
        self.sparsity = float(sparsity)
        self.normalize = bool(normalize)
        self.mean = float(mean)
        self.std = float(std)
        self.fixed_mask_per_sample = bool(fixed_mask_per_sample)
        self.mask_seed = int(mask_seed)
        self.T = int(memmap[int(self.index_pairs[0, 0])].shape[0])

    def __len__(self):
        return len(self.index_pairs)

    def _z(self, a):
        return (a - self.mean) / self.std

    def __getitem__(self, idx):
        item_idx, t = map(int, self.index_pairs[idx])
        seq = self.mem[item_idx]
        x = np.asarray(seq[t], dtype=np.float32)
        y = np.asarray(seq[t + 1], dtype=np.float32)
        if self.normalize:
            x = self._z(x)
            y = self._z(y)
        sid = int(item_idx * self.T + t)
        if self.fixed_mask_per_sample:
            rng = np.random.default_rng(self.mask_seed + sid)
        else:
            rng = np.random.default_rng()
        mask = rng.random(x.shape, dtype=np.float32) < self.sparsity
        x_sparse = x * mask.astype(np.float32)
        x_sparse = torch.from_numpy(x_sparse[None, ...])
        y = torch.from_numpy(y[None, ...])
        mask = torch.from_numpy(mask.astype(np.bool_))
        sample_id = torch.tensor(sid, dtype=torch.long)
        return x_sparse, y, mask, sample_id


mem = RaggedMmap(MMAP_PATH, mode='r')
train_pairs, val_pairs, T = build_fixed_item_splits(mem, TARGET_TRAIN_PAIRS, val_pairs=VAL_PAIRS, train_seed=TRAIN_SEED, val_seed=VAL_SEED)
train_ds = NavierStokesSparseNextStep(mem, train_pairs, sparsity=SPARSITY, normalize=True, mean=MEAN, std=STD, fixed_mask_per_sample=True, mask_seed=0)
val_ds = NavierStokesSparseNextStep(mem, val_pairs, sparsity=SPARSITY, normalize=True, mean=MEAN, std=STD, fixed_mask_per_sample=True, mask_seed=0)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)

label_dim = H * W * 2
conv2d_gradfix.enabled = True


def build_condition(x_sparse, mask):
    mask_f = mask.float().unsqueeze(1)
    x_flat = x_sparse.view(x_sparse.size(0), -1)
    m_flat = mask_f.view(mask_f.size(0), -1)
    return torch.cat([x_flat, m_flat], dim=1)


G = Generator(z_dim=Z_DIM, c_dim=label_dim, w_dim=512, img_resolution=H, img_channels=CHANNELS).to(DEVICE)
D = Discriminator(c_dim=label_dim, img_resolution=H, img_channels=CHANNELS).to(DEVICE)
G_opt = torch.optim.Adam(G.parameters(), lr=LR, betas=BETAS)
D_opt = torch.optim.Adam(D.parameters(), lr=LR, betas=BETAS)


def d_logistic_loss(real_pred, fake_pred):
    return F.softplus(fake_pred).mean() + F.softplus(-real_pred).mean()


def g_nonsat_loss(fake_pred):
    return F.softplus(-fake_pred).mean()


def r1_penalty(real_img, real_pred):
    grad_real = torch.autograd.grad(outputs=real_pred.sum(), inputs=real_img, create_graph=True)[0]
    return grad_real.square().view(real_img.size(0), -1).sum(1).mean()


best_val_mse = float('inf')
global_step = 0

for epoch in tqdm(range(1, EPOCHS + 1), desc='Epochs'):
    G.train(); D.train()
    train_loss_g = 0.0
    train_loss_d = 0.0
    for real_sparse, real_img, mask, _ in train_loader:
        real_sparse = real_sparse.to(DEVICE)
        real_img = real_img.to(DEVICE)
        mask = mask.to(DEVICE)
        cond = build_condition(real_sparse, mask)
        if LABEL_NOISE > 0:
            cond = cond + torch.randn_like(cond) * LABEL_NOISE
        z = torch.randn(real_img.size(0), Z_DIM, device=DEVICE)

        for p in D.parameters():
            p.requires_grad_(True)
        for p in G.parameters():
            p.requires_grad_(False)
        D_opt.zero_grad(set_to_none=True)
        fake_img = G(z, cond)
        fake_pred = D(fake_img.detach(), cond)
        real_pred = D(real_img, cond)
        d_loss = d_logistic_loss(real_pred, fake_pred)
        d_loss.backward()

        global_step += 1
        if global_step % D_REG_INTERVAL == 0:
            real_img.requires_grad_(True)
            real_pred_reg = D(real_img, cond)
            r1 = r1_penalty(real_img, real_pred_reg)
            d_loss_reg = (GAMMA_R1 / 2.0) * r1 * D_REG_INTERVAL
            d_loss_reg.backward()
            real_img = real_img.detach()

        D_opt.step()
        train_loss_d += d_loss.item() * real_img.size(0)

        for p in D.parameters():
            p.requires_grad_(False)
        for p in G.parameters():
            p.requires_grad_(True)
        G_opt.zero_grad(set_to_none=True)
        z = torch.randn(real_img.size(0), Z_DIM, device=DEVICE)
        fake_img = G(z, cond)
        fake_pred = D(fake_img, cond)
        g_loss = g_nonsat_loss(fake_pred)
        g_loss.backward()
        G_opt.step()
        train_loss_g += g_loss.item() * real_img.size(0)

    train_loss_g /= len(train_loader.dataset)
    train_loss_d /= len(train_loader.dataset)

    G.eval(); D.eval()
    val_mse = 0.0
    with torch.no_grad():
        for real_sparse, real_img, mask, _ in val_loader:
            real_sparse = real_sparse.to(DEVICE)
            real_img = real_img.to(DEVICE)
            mask = mask.to(DEVICE)
            cond = build_condition(real_sparse, mask)
            z = torch.randn(real_img.size(0), Z_DIM, device=DEVICE)
            fake_img = G(z, cond)
            val_mse += F.mse_loss(fake_img, real_img, reduction='sum').item()
    val_mse /= len(val_loader.dataset) * CHANNELS

    if val_mse < best_val_mse:
        best_val_mse = val_mse
        torch.save({'G': G.state_dict(), 'D': D.state_dict(), 'epoch': epoch, 'val_mse': val_mse}, 'navierstokes_stylegan3_best.pt')

    print(f"Epoch {epoch:03d} | G_loss={train_loss_g:.4f} | D_loss={train_loss_d:.4f} | Val_MSE={val_mse:.6f} | Best_Val_MSE={best_val_mse:.6f}")

print('Training complete. Best validation MSE:', best_val_mse)
